#### 1. Топы по графам за весь период

In [1]:
#загружаем файлы
from pathlib import Path
import pandas as pd
import numpy as np

gephi_dir = Path("analysis_cursovaya")
files = {file.name: pd.read_excel(file) for file in sorted(gephi_dir.glob("*.xlsx"))}

ct_edges = files["1) channel_topic_edges_metrics.xlsx"]
ct_nodes = files["1) channel_topic_nodes_metrics.xlsx"]
agenda_edges = files["2) channel_channel_agenda_edges_metrics.xlsx"]
agenda_nodes = files["2) channel_channel_agenda_nodes_metrics.xlsx"]
stance_edges = files["3) channel_channel_stance_edges_metrics.xlsx"]
stance_nodes = files["3) channel_channel_stance_nodes_metrics.xlsx"]
tt_edges = files["4) topic_topic_edges_metrics.xlsx"]
tt_nodes = files["4) topic_topic_nodes_metrics.xlsx"]
q1_edges = files["5) channel_channel_stance_2024Q1_edges_metrics.xlsx"]
q1_nodes = files["5) channel_channel_stance_2024Q1_nodes_metrics.xlsx"]
q2_edges = files["6) channel_channel_stance_2025Q2_edges_metrics.xlsx"]
q2_nodes = files["6) channel_channel_stance_2025Q2_nodes_metrics.xlsx"]
q3_edges = files["7) channel_channel_stance_2025Q3_edges_metrics.xlsx"]
q3_nodes = files["7) channel_channel_stance_2025Q3_nodes_metrics.xlsx"]

print("файлы загружены")

файлы загружены


In [2]:
#добавляем рабочее название узла и тип узла
def add_node_name_and_type(df):
    df = df.copy()
    df["node_name"] = df["label"].where(df["label"].notna(), df["id"])
    if "node_type" not in df.columns:
        df["node_type"] = np.where(df["node_name"].astype(str).str.lower().str.contains("topic_"), "topic", "channel")
    return df

ct_nodes = add_node_name_and_type(ct_nodes)
agenda_nodes = add_node_name_and_type(agenda_nodes)
stance_nodes = add_node_name_and_type(stance_nodes)
tt_nodes = add_node_name_and_type(tt_nodes)
q1_nodes = add_node_name_and_type(q1_nodes)
q2_nodes = add_node_name_and_type(q2_nodes)
q3_nodes = add_node_name_and_type(q3_nodes)

ct_nodes["node_type"].value_counts()

node_type
channel    99
topic      19
Name: count, dtype: int64

In [3]:
#задаем единый стиль таблиц, которые будут гармоничны в word
def word_table(df, caption=None, precision=3):
    styler = df.style.format(precision=precision, na_rep="").hide(axis="index")
    styler = styler.set_table_styles([
        {"selector": "table", "props": [("border-collapse", "collapse"), ("font-family", "Times New Roman"), ("font-size", "11pt"), ("width", "100%")]},
        {"selector": "th", "props": [("border-top", "1.5px solid black"), ("border-bottom", "1px solid black"), ("padding", "4px 6px"), ("text-align", "center"), ("font-weight", "bold")]},
        {"selector": "td", "props": [("border-bottom", "0.5px solid #999"), ("padding", "4px 6px"), ("text-align", "center")]},
        {"selector": "td:nth-child(2)", "props": [("text-align", "left")]},
        {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"), ("font-weight", "bold"), ("font-family", "Times New Roman"), ("font-size", "11pt"), ("margin-bottom", "6px")]}
    ])
    return styler.set_caption(caption) if caption else styler

In [4]:
#создаем словарь содержательных названий тем
topic_labels = (
    ct_edges[["target", "target_label"]]
    .dropna()
    .drop_duplicates()
    .set_index("target")["target_label"]
    .to_dict()
)

topic_labels

{'topic_0': 'topic_0: международная политика / Украина / США',
 'topic_1': 'topic_1: боевые действия / фронт / военные сводки',
 'topic_2': 'topic_2: экономика / рынок / энергетика',
 'topic_3': 'topic_3: ПВО / Курская область / ракетные угрозы',
 'topic_4': 'topic_4: культура / кино / медиа',
 'topic_5': 'topic_5: цифровая среда / мошенничество / платформы',
 'topic_6': 'topic_6: Украина / регионы / церковь',
 'topic_7': 'topic_7: спорт',
 'topic_11': 'topic_11: энергетика / аварии / отключения',
 'topic_10': 'topic_10: здравоохранение / медицина',
 'topic_9': 'topic_9: образование / школы / студенты',
 'topic_8': 'topic_8: искусственный интеллект / технологии',
 'topic_13': 'topic_13: семья / рождаемость / аборты',
 'topic_12': 'topic_12: авиация / аэропорты / ограничения',
 'topic_16': 'topic_16: иноагенты / политические ограничения',
 'topic_14': 'topic_14: алкоголь / вейпы / регулирование',
 'topic_15': 'topic_15: погода / температура',
 'topic_18': 'topic_18: пожертвования / подд

In [5]:
#добавляем содержательные названия тем в таблицы с темами
def add_topic_label(df):
    df = df.copy()
    df["topic_label"] = df["node_name"].map(topic_labels)
    df["node_readable"] = np.where(df["topic_label"].notna(), df["topic_label"], df["node_name"])
    return df

ct_nodes = add_topic_label(ct_nodes)
tt_nodes = add_topic_label(tt_nodes)

In [6]:
#собираем топ-10 узлов 
def top_nodes(df, metric, node_type=None, n=10):
    data = df.copy()
    if node_type:
        data = data[data["node_type"] == node_type]
    name_col = "node_readable" if "node_readable" in data.columns else "node_name"
    metrics = ["degree", "weighted_degree", "pagerank", "betweenness", "eigenvector", "modularity_class"]
    metrics = [m for m in metrics if m != metric and m in data.columns]
    cols = [name_col, metric] + metrics
    out = data.sort_values(metric, ascending=False).head(n)[cols].reset_index(drop=True)
    out.insert(0, "№", range(1, len(out) + 1))
    return out.rename(columns={
        name_col: "Узел",
        "degree": "Степень",
        "weighted_degree": "Взвешенная степень",
        "pagerank": "PageRank",
        "betweenness": "Посредническая центральность",
        "eigenvector": "Собственная центральность",
        "modularity_class": "Класс модулярности"
    })

In [7]:
#выводим топ-10 каналов в channel-topic сети по взвешенной степени
ct_top_channels = top_nodes(ct_nodes, "weighted_degree", node_type="channel")
word_table(ct_top_channels, "Таблица 1. Топ-10 каналов в сети channel-topic по взвешенной степени")

№,Узел,Взвешенная степень,Степень,PageRank,Посредническая центральность,Собственная центральность,Класс модулярности
1,@mnogonazi,0.992,10,0.008,39.259,0.299,0
2,@government_rus,0.990,11,0.008,41.538,0.286,0
3,@razvozhaev,0.985,10,0.007,20.771,0.285,1
4,@golosmordora1,0.985,10,0.007,16.666,0.313,0
5,@obshina_ru,0.983,9,0.008,131.616,0.263,0
6,@polittemnik,0.982,10,0.008,23.441,0.299,0
7,@sanya_florida,0.980,9,0.007,13.751,0.287,1
8,@mardanaka,0.978,13,0.009,42.313,0.348,0
9,@gubernator_46,0.978,4,0.004,1.483,0.157,1
10,@boris_rozhin,0.978,8,0.006,7.503,0.277,1


In [8]:
#выводим топ-10 тем в channel-topic сети по взвешенной степени
ct_top_topics = top_nodes(ct_nodes, "weighted_degree", node_type="topic")
word_table(ct_top_topics, "Таблица 2. Топ-10 тем в сети channel-topic по взвешенной степени")

№,Узел,Взвешенная степень,Степень,PageRank,Посредническая центральность,Собственная центральность,Класс модулярности
1,topic_0: международная политика / Украина / США,47.151,98,0.077,2117.244,1.000,0
2,topic_1: боевые действия / фронт / военные сводки,12.753,77,0.056,952.174,0.869,1
3,topic_2: экономика / рынок / энергетика,9.190,77,0.056,902.192,0.876,0
4,topic_4: культура / кино / медиа,6.813,70,0.051,728.039,0.811,2
5,topic_3: ПВО / Курская область / ракетные угрозы,3.332,42,0.029,184.510,0.542,1
6,topic_5: цифровая среда / мошенничество / платформы,3.141,60,0.043,501.170,0.709,0
7,topic_6: Украина / регионы / церковь,2.448,45,0.032,262.503,0.541,2
8,topic_7: спорт,1.676,43,0.030,207.415,0.549,0
9,topic_9: образование / школы / студенты,1.469,25,0.018,68.571,0.317,3
10,topic_8: искусственный интеллект / технологии,0.934,36,0.025,137.096,0.466,0


In [9]:
#выводим топ-10 каналов в agenda-сети по взвешенной степени
agenda_top_channels = top_nodes(agenda_nodes, "weighted_degree", node_type="channel")
word_table(agenda_top_channels, "Таблица 3. Топ-10 каналов в agenda-сети по взвешенной степени")

№,Узел,Взвешенная степень,Степень,PageRank,Посредническая центральность,Собственная центральность,Класс модулярности
1,@dmitrynikotin,31.987,33,0.020,287.761,1.000,1
2,@slvn_pomet,26.037,27,0.017,180.397,0.855,1
3,@vatnoeboloto,22.717,23,0.014,251.608,0.698,4
4,@rlz_the_kraken,22.704,23,0.014,224.222,0.718,1
5,@Taynaya_kantselyariya,20.820,21,0.013,73.857,0.625,1
6,@ejdailyru,20.661,21,0.014,83.403,0.539,4
7,@mardanaka,19.735,20,0.013,200.483,0.676,1
8,@EvPanina,18.828,19,0.012,65.793,0.565,1
9,@obrazbuduschego2,18.513,19,0.012,59.561,0.503,4
10,@ASGasparyan,18.472,20,0.013,108.952,0.589,3


In [10]:
#выводим топ-10 каналов в stance-сети по pagerank
stance_top_pagerank = top_nodes(stance_nodes, "pagerank", node_type="channel")
word_table(stance_top_pagerank, "Таблица 4. Топ-10 каналов в stance-сети по PageRank")

№,Узел,PageRank,Степень,Взвешенная степень,Посредническая центральность,Собственная центральность,Класс модулярности
1,@Putin_tramp_mobilizaciya_migrant,0.041,73,10.949,547.178,1.000,0
2,@khodorkovski,0.040,70,10.694,609.661,0.900,1
3,@ironlogica,0.038,67,11.275,507.966,0.861,0
4,@skabeeva,0.038,68,9.359,387.282,0.962,0
5,@mnogonazi,0.036,64,9.618,425.085,0.844,1
6,@Taynaya_kantselyariya,0.031,55,7.504,206.566,0.792,0
7,@steven_bb,0.028,49,6.676,159.453,0.721,1
8,@protiktok,0.025,43,6.671,137.949,0.587,0
9,@slvn_pomet,0.024,42,5.602,119.664,0.583,1
10,@aavst2022,0.024,41,5.359,116.200,0.566,1


In [11]:
#выводим топ-10 тем в topic-topic сети по взвешенной степени
tt_top_weighted = top_nodes(tt_nodes, "weighted_degree", node_type="topic")
word_table(tt_top_weighted, "Таблица 5. Топ-10 тем в topic-topic сети по взвешенной степени")

№,Узел,Взвешенная степень,Степень,PageRank,Посредническая центральность,Собственная центральность,Класс модулярности
1,topic_0: международная политика / Украина / США,5.905,13,0.085,28.451,1.000,2
2,topic_5: цифровая среда / мошенничество / платформы,4.834,9,0.062,9.061,0.791,1
3,topic_8: искусственный интеллект / технологии,4.589,9,0.063,9.938,0.758,1
4,topic_7: спорт,4.516,10,0.069,11.998,0.796,2
5,topic_10: здравоохранение / медицина,4.002,9,0.064,9.079,0.718,0
6,topic_2: экономика / рынок / энергетика,3.949,9,0.063,9.560,0.751,0
7,topic_13: семья / рождаемость / аборты,3.644,9,0.059,25.510,0.630,1
8,topic_12: авиация / аэропорты / ограничения,2.732,6,0.044,1.017,0.581,0
9,topic_18: пожертвования / поддержка организаций,2.721,5,0.025,0.000,0.320,1
10,topic_14: алкоголь / вейпы / регулирование,2.720,5,0.037,0.327,0.470,1


In [12]:
#выводим топ-10 тем в topic-topic сети по посреднической центральности
tt_top_betweenness = top_nodes(tt_nodes, "betweenness", node_type="topic")
word_table(tt_top_betweenness, "Таблица 6. Топ-10 тем в topic-topic сети по посреднической центральности")

№,Узел,Посредническая центральность,Степень,Взвешенная степень,PageRank,Собственная центральность,Класс модулярности
1,topic_0: международная политика / Украина / США,28.451,13,5.905,0.085,1.000,2
2,topic_13: семья / рождаемость / аборты,25.510,9,3.644,0.059,0.630,1
3,topic_7: спорт,11.998,10,4.516,0.069,0.796,2
4,topic_8: искусственный интеллект / технологии,9.938,9,4.589,0.063,0.758,1
5,topic_2: экономика / рынок / энергетика,9.560,9,3.949,0.063,0.751,0
6,topic_10: здравоохранение / медицина,9.079,9,4.002,0.064,0.718,0
7,topic_5: цифровая среда / мошенничество / платформы,9.061,9,4.834,0.062,0.791,1
8,topic_6: Украина / регионы / церковь,4.884,6,2.299,0.042,0.496,2
9,topic_11: энергетика / аварии / отключения,3.529,7,2.642,0.051,0.584,2
10,topic_1: боевые действия / фронт / военные сводки,2.894,6,2.492,0.044,0.524,2


In [13]:
#собираем топ-10 ребер по выбранной метрике
def top_edges(df, metric, n=10, edge_type=None):
    data = df.copy()
    if edge_type and "edge_type" in data.columns:
        data = data[data["edge_type"] == edge_type]
    cols = ["source", "target", metric, "weight", "similarity", "signed_weight", "abs_signed_weight", "edge_type", "avg_sentiment", "dominant_stance"]
    cols = [c for i, c in enumerate(cols) if c in data.columns and c not in cols[:i]]
    out = data.sort_values(metric, ascending=False).head(n)[cols].reset_index(drop=True)
    out.insert(0, "№", range(1, len(out) + 1))
    return out.rename(columns={
        "source": "Источник",
        "target": "Цель",
        "weight": "Вес",
        "similarity": "Сходство",
        "signed_weight": "Знаковый вес",
        "abs_signed_weight": "Сила связи",
        "edge_type": "Тип связи",
        "avg_sentiment": "Средний sentiment",
        "dominant_stance": "Доминирующая позиция"
    })

In [14]:
#выводим топ-10 связей канал-тема по весу
ct_top_edges = top_edges(ct_edges, "weight")
word_table(ct_top_edges, "Таблица 7. Топ-10 связей канал–тема по весу")

№,Источник,Цель,Вес,Средний sentiment,Доминирующая позиция
1,@mod_russia,topic_1,0.871,-0.012,neutral
2,@medvedev_telegram,topic_0,0.833,-0.177,negative
3,@gubernator_46,topic_3,0.812,-0.044,neutral
4,@skabeeva,topic_0,0.800,-0.172,negative
5,@MariaVladimirovnaZakharova,topic_0,0.793,-0.093,neutral
6,@ironlogica,topic_0,0.782,-0.128,neutral
7,@Taynaya_kantselyariya,topic_0,0.776,-0.155,negative
8,@obzorpolita,topic_0,0.770,-0.031,neutral
9,@khodorkovski,topic_0,0.767,-0.459,negative
10,@EvPanina,topic_0,0.762,-0.130,negative


In [15]:
#выводим топ-10 пар каналов в agenda-сети по тематическому сходству
agenda_top_edges = top_edges(agenda_edges, "similarity")
word_table(agenda_top_edges, "Таблица 8. Топ-10 пар каналов в agenda-сети по тематическому сходству")

№,Источник,Цель,Сходство,Вес
1,@dmitrynikotin,@slvn_pomet,0.999,0.999
2,@EvPanina,@Taynaya_kantselyariya,0.998,0.998
3,@generalsvr,@steven_bb,0.998,0.998
4,@dmitrynikotin,@rlz_the_kraken,0.998,0.998
5,@Putin_tramp_mobilizaciya_migrant,@skabeeva,0.998,0.998
6,@RVvoenkor,@sanya_florida,0.998,0.998
7,@MariaVladimirovnaZakharova,@skabeeva,0.997,0.997
8,@Putin_tramp_mobilizaciya_migrant,@obzorpolita,0.997,0.997
9,@Rrussiansword,@ironlogica,0.997,0.997
10,@ejdailyru,@politjoystic,0.997,0.997


In [16]:
#выводим топ-10 пар каналов в stance-сети по силе тонально-тематической связи
stance_top_edges = top_edges(stance_edges, "abs_signed_weight")
word_table(stance_top_edges, "Таблица 9. Топ-10 пар каналов в stance-сети по силе связи")

№,Источник,Цель,Сила связи,Вес,Знаковый вес,Тип связи
1,@khodorkovski,@mnogonazi,0.311,0.311,0.311,congruence
2,@government_rus,@mkhusnullin,0.310,0.310,0.310,congruence
3,@ironlogica,@protiktok,0.307,0.307,0.307,congruence
4,@ironlogica,@obzorpolita,0.303,0.303,0.303,congruence
5,@Putin_tramp_mobilizaciya_migrant,@ironlogica,0.297,0.297,0.297,congruence
6,@obzorpolita,@protiktok,0.262,0.262,0.262,congruence
7,@ironlogica,@news_kremlin,0.258,0.258,0.258,congruence
8,@ironlogica,@polit_inform,0.256,0.256,0.256,congruence
9,@ironlogica,@khazinml,0.249,0.249,0.249,congruence
10,@khodorkovski,@slvn_pomet,0.249,0.249,0.249,congruence


In [17]:
#проверяем распределение типов связей в stance-сети
stance_edge_types = stance_edges["edge_type"].value_counts().reset_index()
stance_edge_types.columns = ["Тип связи", "Число ребер"]
stance_edge_types["Доля"] = stance_edge_types["Число ребер"] / stance_edge_types["Число ребер"].sum()
word_table(stance_edge_types, "Таблица 10. Распределение типов связей в stance-сети")

Тип связи,Число ребер,Доля
congruence,770,0.978
conflict,17,0.022


#### 2. Топы по графам за избранные кварталы

In [18]:
#собираем квартальные таблицы узлов в один словарь
quarter_nodes = {
    "2024Q1": q1_nodes,
    "2025Q2": q2_nodes,
    "2025Q3": q3_nodes
}

quarter_edges = {
    "2024Q1": q1_edges,
    "2025Q2": q2_edges,
    "2025Q3": q3_edges
}

In [19]:
#выводим топ-10 каналов по pagerank для каждого квартала
for quarter, df in quarter_nodes.items():
    table = top_nodes(df, "pagerank", node_type="channel")
    display(word_table(table, f"Таблица. Топ-10 каналов в stance-сети {quarter} по PageRank"))

№,Узел,PageRank,Степень,Взвешенная степень,Посредническая центральность,Собственная центральность,Класс модулярности
1,@khodorkovski,0.040,74,17.903,422.962,1.000,0
2,@kremlin_bulldogs,0.039,72,15.309,358.840,0.991,0
3,@medvedev_telegram,0.039,71,18.472,514.244,0.921,0
4,@MariaVladimirovnaZakharova,0.037,68,14.183,318.905,0.931,1
5,@Rrussiansword,0.036,66,13.537,288.706,0.913,1
6,@ironlogica,0.035,63,13.158,291.330,0.852,1
7,@satirkka,0.034,63,12.526,258.049,0.902,0
8,@mnogonazi,0.028,51,10.889,135.490,0.769,0
9,@slvn_pomet,0.028,50,11.161,173.036,0.668,0
10,@Taynaya_kantselyariya,0.026,48,9.499,101.089,0.755,1


№,Узел,PageRank,Степень,Взвешенная степень,Посредническая центральность,Собственная центральность,Класс модулярности
1,@eschulmann,0.037,67,10.465,523.167,1.000,0
2,@slvn_pomet,0.034,62,9.581,365.876,0.924,1
3,@medvedev_telegram,0.033,59,13.163,389.261,0.856,0
4,@MDaudov_95,0.031,55,10.711,345.039,0.783,1
5,@ironlogica,0.030,54,8.369,257.508,0.817,0
6,@mnogonazi,0.029,51,8.430,242.109,0.734,1
7,@Putin_tramp_mobilizaciya_migrant,0.028,51,7.298,183.903,0.832,0
8,@dshrg2,0.028,49,7.243,237.962,0.732,1
9,@novosty937,0.023,41,5.681,132.230,0.671,0
10,@protiktok,0.022,38,5.635,124.737,0.564,0


№,Узел,PageRank,Степень,Взвешенная степень,Посредническая центральность,Собственная центральность,Класс модулярности
1,@strelkovii,0.041,78,16.618,584.706,1.000,1
2,@ironlogica,0.039,74,14.545,468.253,0.964,1
3,@khazinml,0.036,68,11.826,359.933,0.914,1
4,@medvedev_telegram,0.035,66,16.010,449.802,0.834,0
5,@Taynaya_kantselyariya,0.034,65,11.797,313.142,0.843,1
6,@skabeeva,0.030,57,9.422,183.660,0.813,1
7,@khodorkovski,0.028,52,8.745,297.663,0.653,0
8,@martynov2021,0.028,52,8.338,178.777,0.721,1
9,@mnogonazi,0.027,51,8.623,169.388,0.696,0
10,@Putin_tramp_mobilizaciya_migrant,0.024,46,7.503,99.554,0.671,1


In [20]:
#выводим топ-10 каналов по взвешенной степени для каждого квартала
for quarter, df in quarter_nodes.items():
    table = top_nodes(df, "weighted_degree", node_type="channel")
    display(word_table(table, f"Таблица. Топ-10 каналов в stance-сети {quarter} по взвешенной степени"))

№,Узел,Взвешенная степень,Степень,PageRank,Посредническая центральность,Собственная центральность,Класс модулярности
1,@medvedev_telegram,18.472,71,0.039,514.244,0.921,0
2,@khodorkovski,17.903,74,0.040,422.962,1.000,0
3,@kremlin_bulldogs,15.309,72,0.039,358.840,0.991,0
4,@MariaVladimirovnaZakharova,14.183,68,0.037,318.905,0.931,1
5,@Rrussiansword,13.537,66,0.036,288.706,0.913,1
6,@ironlogica,13.158,63,0.035,291.330,0.852,1
7,@satirkka,12.526,63,0.034,258.049,0.902,0
8,@slvn_pomet,11.161,50,0.028,173.036,0.668,0
9,@mnogonazi,10.889,51,0.028,135.490,0.769,0
10,@Taynaya_kantselyariya,9.499,48,0.026,101.089,0.755,1


№,Узел,Взвешенная степень,Степень,PageRank,Посредническая центральность,Собственная центральность,Класс модулярности
1,@medvedev_telegram,13.163,59,0.033,389.261,0.856,0
2,@MDaudov_95,10.711,55,0.031,345.039,0.783,1
3,@eschulmann,10.465,67,0.037,523.167,1.000,0
4,@slvn_pomet,9.581,62,0.034,365.876,0.924,1
5,@mnogonazi,8.430,51,0.029,242.109,0.734,1
6,@ironlogica,8.369,54,0.030,257.508,0.817,0
7,@Putin_tramp_mobilizaciya_migrant,7.298,51,0.028,183.903,0.832,0
8,@dshrg2,7.243,49,0.028,237.962,0.732,1
9,@novosty937,5.681,41,0.023,132.230,0.671,0
10,@protiktok,5.635,38,0.022,124.737,0.564,0


№,Узел,Взвешенная степень,Степень,PageRank,Посредническая центральность,Собственная центральность,Класс модулярности
1,@strelkovii,16.618,78,0.041,584.706,1.000,1
2,@medvedev_telegram,16.010,66,0.035,449.802,0.834,0
3,@ironlogica,14.545,74,0.039,468.253,0.964,1
4,@khazinml,11.826,68,0.036,359.933,0.914,1
5,@Taynaya_kantselyariya,11.797,65,0.034,313.142,0.843,1
6,@skabeeva,9.422,57,0.030,183.660,0.813,1
7,@khodorkovski,8.745,52,0.028,297.663,0.653,0
8,@mnogonazi,8.623,51,0.027,169.388,0.696,0
9,@martynov2021,8.338,52,0.028,178.777,0.721,1
10,@Putin_tramp_mobilizaciya_migrant,7.503,46,0.024,99.554,0.671,1


In [21]:
#собираем сравнительную таблицу топов по pagerank между кварталами
quarter_top_pagerank = []

for quarter, df in quarter_nodes.items():
    top = top_nodes(df, "pagerank", node_type="channel")
    top["Квартал"] = quarter
    top["Метрика"] = "PageRank"
    quarter_top_pagerank.append(top)

quarter_top_pagerank = pd.concat(quarter_top_pagerank, ignore_index=True)
quarter_top_pagerank = quarter_top_pagerank[["Квартал", "№", "Узел", "PageRank", "Степень", "Взвешенная степень", "Посредническая центральность", "Собственная центральность", "Класс модулярности"]]

word_table(quarter_top_pagerank, "Таблица. Топ-10 каналов по PageRank в квартальных stance-сетях")

Квартал,№,Узел,PageRank,Степень,Взвешенная степень,Посредническая центральность,Собственная центральность,Класс модулярности
2024Q1,1,@khodorkovski,0.040,74,17.903,422.962,1.000,0
2024Q1,2,@kremlin_bulldogs,0.039,72,15.309,358.840,0.991,0
2024Q1,3,@medvedev_telegram,0.039,71,18.472,514.244,0.921,0
2024Q1,4,@MariaVladimirovnaZakharova,0.037,68,14.183,318.905,0.931,1
2024Q1,5,@Rrussiansword,0.036,66,13.537,288.706,0.913,1
2024Q1,6,@ironlogica,0.035,63,13.158,291.330,0.852,1
2024Q1,7,@satirkka,0.034,63,12.526,258.049,0.902,0
2024Q1,8,@mnogonazi,0.028,51,10.889,135.490,0.769,0
2024Q1,9,@slvn_pomet,0.028,50,11.161,173.036,0.668,0
2024Q1,10,@Taynaya_kantselyariya,0.026,48,9.499,101.089,0.755,1


In [22]:
#собираем сравнительную таблицу топов по взвешенной степени между кварталами
quarter_top_weighted = []

for quarter, df in quarter_nodes.items():
    top = top_nodes(df, "weighted_degree", node_type="channel")
    top["Квартал"] = quarter
    top["Метрика"] = "Взвешенная степень"
    quarter_top_weighted.append(top)

quarter_top_weighted = pd.concat(quarter_top_weighted, ignore_index=True)
quarter_top_weighted = quarter_top_weighted[["Квартал", "№", "Узел", "Взвешенная степень", "Степень", "PageRank", "Посредническая центральность", "Собственная центральность", "Класс модулярности"]]

word_table(quarter_top_weighted, "Таблица. Топ-10 каналов по взвешенной степени в квартальных stance-сетях")

Квартал,№,Узел,Взвешенная степень,Степень,PageRank,Посредническая центральность,Собственная центральность,Класс модулярности
2024Q1,1,@medvedev_telegram,18.472,71,0.039,514.244,0.921,0
2024Q1,2,@khodorkovski,17.903,74,0.040,422.962,1.000,0
2024Q1,3,@kremlin_bulldogs,15.309,72,0.039,358.840,0.991,0
2024Q1,4,@MariaVladimirovnaZakharova,14.183,68,0.037,318.905,0.931,1
2024Q1,5,@Rrussiansword,13.537,66,0.036,288.706,0.913,1
2024Q1,6,@ironlogica,13.158,63,0.035,291.330,0.852,1
2024Q1,7,@satirkka,12.526,63,0.034,258.049,0.902,0
2024Q1,8,@slvn_pomet,11.161,50,0.028,173.036,0.668,0
2024Q1,9,@mnogonazi,10.889,51,0.028,135.490,0.769,0
2024Q1,10,@Taynaya_kantselyariya,9.499,48,0.026,101.089,0.755,1


In [23]:
#считаем, какие каналы чаще всего попадают в квартальные топ-10 по pagerank
pagerank_recurrence = (
    quarter_top_pagerank
    .groupby("Узел")
    .agg(
        Количество_кварталов=("Квартал", "nunique"),
        Кварталы=("Квартал", lambda x: ", ".join(sorted(x.unique()))),
        Средний_PageRank=("PageRank", "mean"),
        Средний_ранг=("№", "mean")
    )
    .reset_index()
    .sort_values(["Количество_кварталов", "Средний_ранг"], ascending=[False, True])
)

word_table(pagerank_recurrence, "Таблица. Повторяемость каналов в квартальных топ-10 по PageRank")

Узел,Количество_кварталов,Кварталы,Средний_PageRank,Средний_ранг
@medvedev_telegram,3,"2024Q1, 2025Q2, 2025Q3",0.036,3.333
@ironlogica,3,"2024Q1, 2025Q2, 2025Q3",0.034,4.333
@mnogonazi,3,"2024Q1, 2025Q2, 2025Q3",0.028,7.667
@khodorkovski,2,"2024Q1, 2025Q3",0.034,4.000
@slvn_pomet,2,"2024Q1, 2025Q2",0.031,5.500
@Taynaya_kantselyariya,2,"2024Q1, 2025Q3",0.030,7.500
@Putin_tramp_mobilizaciya_migrant,2,"2025Q2, 2025Q3",0.026,8.500
@eschulmann,1,2025Q2,0.037,1.000
@strelkovii,1,2025Q3,0.041,1.000
@kremlin_bulldogs,1,2024Q1,0.039,2.000


In [24]:
#считаем, какие каналы чаще всего попадают в квартальные топ-10 по взвешенной степени
weighted_recurrence = (
    quarter_top_weighted
    .groupby("Узел")
    .agg(
        Количество_кварталов=("Квартал", "nunique"),
        Кварталы=("Квартал", lambda x: ", ".join(sorted(x.unique()))),
        Средняя_взвешенная_степень=("Взвешенная степень", "mean"),
        Средний_ранг=("№", "mean")
    )
    .reset_index()
    .sort_values(["Количество_кварталов", "Средний_ранг"], ascending=[False, True])
)

word_table(weighted_recurrence, "Таблица. Повторяемость каналов в квартальных топ-10 по взвешенной степени")

Узел,Количество_кварталов,Кварталы,Средняя_взвешенная_степень,Средний_ранг
@medvedev_telegram,3,"2024Q1, 2025Q2, 2025Q3",15.882,1.333
@ironlogica,3,"2024Q1, 2025Q2, 2025Q3",12.024,5.000
@mnogonazi,3,"2024Q1, 2025Q2, 2025Q3",9.314,7.333
@khodorkovski,2,"2024Q1, 2025Q3",13.324,4.500
@slvn_pomet,2,"2024Q1, 2025Q2",10.371,6.000
@Taynaya_kantselyariya,2,"2024Q1, 2025Q3",10.648,7.500
@Putin_tramp_mobilizaciya_migrant,2,"2025Q2, 2025Q3",7.400,8.500
@strelkovii,1,2025Q3,16.618,1.000
@MDaudov_95,1,2025Q2,10.711,2.000
@eschulmann,1,2025Q2,10.465,3.000


In [25]:
#выводим топ-10 пар каналов по силе stance-связи для каждого квартала
for quarter, df in quarter_edges.items():
    table = top_edges(df, "abs_signed_weight")
    display(word_table(table, f"Таблица. Топ-10 пар каналов в stance-сети {quarter} по силе связи"))

№,Источник,Цель,Сила связи,Вес,Знаковый вес,Тип связи
1,@medvedev_telegram,@slvn_pomet,0.689,0.689,-0.689,conflict
2,@khodorkovski,@medvedev_telegram,0.667,0.667,-0.667,conflict
3,@medvedev_telegram,@mnogonazi,0.544,0.544,-0.544,conflict
4,@kremlin_bulldogs,@medvedev_telegram,0.500,0.500,-0.500,conflict
5,@khodorkovski,@slvn_pomet,0.481,0.481,0.481,congruence
6,@khodorkovski,@kremlin_bulldogs,0.458,0.458,0.458,congruence
7,@khodorkovski,@mnogonazi,0.433,0.433,0.433,congruence
8,@auantonov,@medvedev_telegram,0.429,0.429,-0.429,conflict
9,@medvedev_telegram,@satirkka,0.412,0.412,-0.412,conflict
10,@medvedev_telegram,@politjoystic,0.410,0.410,-0.410,conflict


№,Источник,Цель,Сила связи,Вес,Знаковый вес,Тип связи
1,@medvedev_telegram,@protiktok,0.506,0.506,0.506,congruence
2,@ironlogica,@medvedev_telegram,0.500,0.500,0.500,congruence
3,@Taynaya_kantselyariya,@medvedev_telegram,0.438,0.438,0.438,congruence
4,@MDaudov_95,@mnogonazi,0.432,0.432,-0.432,conflict
5,@MDaudov_95,@slvn_pomet,0.375,0.375,-0.375,conflict
6,@medvedev_telegram,@novosty937,0.373,0.373,0.373,congruence
7,@ejdailyru,@medvedev_telegram,0.365,0.365,0.365,congruence
8,@eschulmann,@medvedev_telegram,0.364,0.364,0.364,congruence
9,@EvPanina,@medvedev_telegram,0.361,0.361,0.361,congruence
10,@Putin_tramp_mobilizaciya_migrant,@medvedev_telegram,0.360,0.360,0.360,congruence


№,Источник,Цель,Сила связи,Вес,Знаковый вес,Тип связи
1,@khodorkovski,@medvedev_telegram,0.556,0.556,0.556,congruence
2,@medvedev_telegram,@mnogonazi,0.500,0.500,0.500,congruence
3,@medvedev_telegram,@strelkovii,0.500,0.500,0.500,congruence
4,@eschulmann,@medvedev_telegram,0.455,0.455,0.455,congruence
5,@ironlogica,@strelkovii,0.444,0.444,0.444,congruence
6,@medvedev_telegram,@vv_volodin,0.429,0.429,0.429,congruence
7,@Russiacalling,@medvedev_telegram,0.429,0.429,0.429,congruence
8,@Taynaya_kantselyariya,@strelkovii,0.396,0.396,0.396,congruence
9,@Taynaya_kantselyariya,@ironlogica,0.394,0.394,0.394,congruence
10,@khazinml,@strelkovii,0.375,0.375,0.375,congruence


In [26]:
#сравниваем распределение congruence и conflict в квартальных stance-сетях
quarter_edge_types = []

for quarter, df in quarter_edges.items():
    counts = df["edge_type"].value_counts().reset_index()
    counts.columns = ["Тип связи", "Число ребер"]
    counts["Квартал"] = quarter
    counts["Доля"] = counts["Число ребер"] / counts["Число ребер"].sum()
    quarter_edge_types.append(counts)

quarter_edge_types = pd.concat(quarter_edge_types, ignore_index=True)
quarter_edge_types = quarter_edge_types[["Квартал", "Тип связи", "Число ребер", "Доля"]]

word_table(quarter_edge_types, "Таблица. Распределение типов связей в квартальных stance-сетях")

Квартал,Тип связи,Число ребер,Доля
2024Q1,congruence,733,0.878
2024Q1,conflict,102,0.122
2025Q2,congruence,715,0.884
2025Q2,conflict,94,0.116
2025Q3,congruence,805,0.932
2025Q3,conflict,59,0.068


#### 3. Описание топиков и каналов

In [27]:
from pathlib import Path
import pandas as pd

data_dir = Path("analysis_cursovaya")

print(data_dir.exists())
print(data_dir.resolve())

True
C:\Users\sonya\analysis_cursovaya


In [28]:
#функция поиска файла по началу названия (из-за проблем с расширением)
def find_file(stem):
    matches = sorted(data_dir.glob(stem + "*"))
    if len(matches) == 0:
        return None
    return matches[0]

targets = [
    "doc_topic_sentiment_df_giga_r20_50k_100k",
    "final_bertopic_top_words_giga_r20_50k",
    "final_bertopic_topic_info_giga_r20_50k",
    "A_neg_all_giga_r20_50k_100k",
    "A_neu_all_giga_r20_50k_100k",
    "A_pos_all_giga_r20_50k_100k",
    "C_all_giga_r20_50k_100k",
    "controls_all_giga_r20_50k_100k",
    "P_all_giga_r20_50k_100k",
    "S_all_giga_r20_50k_100k"
]

for stem in targets:
    print(stem, "->", find_file(stem))

doc_topic_sentiment_df_giga_r20_50k_100k -> analysis_cursovaya\doc_topic_sentiment_df_giga_r20_50k_100k.parquet
final_bertopic_top_words_giga_r20_50k -> analysis_cursovaya\final_bertopic_top_words_giga_r20_50k.xls
final_bertopic_topic_info_giga_r20_50k -> analysis_cursovaya\final_bertopic_topic_info_giga_r20_50k.xls
A_neg_all_giga_r20_50k_100k -> analysis_cursovaya\A_neg_all_giga_r20_50k_100k.xls
A_neu_all_giga_r20_50k_100k -> analysis_cursovaya\A_neu_all_giga_r20_50k_100k.xls
A_pos_all_giga_r20_50k_100k -> analysis_cursovaya\A_pos_all_giga_r20_50k_100k.xls
C_all_giga_r20_50k_100k -> analysis_cursovaya\C_all_giga_r20_50k_100k.xls
controls_all_giga_r20_50k_100k -> analysis_cursovaya\controls_all_giga_r20_50k_100k.xls
P_all_giga_r20_50k_100k -> analysis_cursovaya\P_all_giga_r20_50k_100k.xls
S_all_giga_r20_50k_100k -> analysis_cursovaya\S_all_giga_r20_50k_100k.xls


In [29]:
#создаем функцию чтения таблиц с разными реальными форматами
def read_table(path):
    path = Path(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".xlsx":
        return pd.read_excel(path)
    for sep in ["\t", ",", ";"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] > 1:
                return df
        except Exception:
            pass
    try:
        return pd.read_excel(path)
    except Exception:
        pass
    try:
        return pd.read_html(path)[0]
    except Exception as e:
        raise ValueError(f"не удалось прочитать файл: {path.name}") from e

In [30]:
#загружаем найденные bertopic-таблицы
doc_topic_sentiment = read_table(find_file("doc_topic_sentiment_df_giga_r20_50k_100k"))
topic_info = read_table(find_file("final_bertopic_topic_info_giga_r20_50k"))
top_words = read_table(find_file("final_bertopic_top_words_giga_r20_50k"))

C = read_table(find_file("C_all_giga_r20_50k_100k"))
P = read_table(find_file("P_all_giga_r20_50k_100k"))
S = read_table(find_file("S_all_giga_r20_50k_100k"))
A_pos = read_table(find_file("A_pos_all_giga_r20_50k_100k"))
A_neg = read_table(find_file("A_neg_all_giga_r20_50k_100k"))
A_neu = read_table(find_file("A_neu_all_giga_r20_50k_100k"))
controls = read_table(find_file("controls_all_giga_r20_50k_100k"))

In [31]:
#готовим справочник тем bertopic
topic_counts_100k = (
    doc_topic_sentiment["topic_id"]
    .value_counts()
    .rename_axis("topic_id")
    .reset_index(name="topic_size_100k")
)

topic_reference = top_words[["topic_id", "topic_name_manual", "top_words"]].copy()
topic_reference = topic_reference.merge(topic_counts_100k, on="topic_id", how="left")

topic_reference["topic_size_100k"] = topic_reference["topic_size_100k"].fillna(0).astype(int)

topic_reference = topic_reference[["topic_id", "topic_size_100k", "topic_name_manual", "top_words"]].copy()
topic_reference = topic_reference.rename(columns={
    "topic_id": "№ темы",
    "topic_size_100k": "Размер темы",
    "topic_name_manual": "Название темы",
    "top_words": "Ключевые слова"
})

topic_reference = topic_reference.sort_values("№ темы").reset_index(drop=True)

word_table(topic_reference, "Таблица. Справочник тем BERTopic", precision=0)

№ темы,Размер темы,Название темы,Ключевые слова
0,30939,международная политика / Украина / США,"россии, украине, заявил, трамп, украины, трампа, нато, президент, президента, путин, россия, израиля"
1,11467,боевые действия / фронт / военные сводки,"противника, группировки, войск, направлении, минобороны, группировки войск, пво, минобороны россии, бригады, противник, военнослужащих, бригад"
2,5688,экономика / рынок / энергетика,"россии, компании, рынок, нефти, экономики, роста, тыс, рынка, цены, развития, цен, ставки"
3,3343,ПВО / Курская область / ракетные угрозы,"курская, опасности, отбой, ракетной опасности, опасность, ракетной, отбой ракетной, внимание отбой, ракетная опасность, ракетная, курская внимание, сплошными"
4,2767,культура / кино / медиа,"фильм, кино, русский формат, русский, формат, концерт, фильма, песни, дабл ять, дабл, ять, культуры"
5,2204,цифровая среда / мошенничество / платформы,"маск, мошенники, дуров, дурова, мошенников, илон, илон маск, доступ, собак, интернета, маска, пользователей"
6,1371,Украина / регионы / церковь,"сумская, харьковская, донбасс, церкви, россия мощь, сила россия, мощь, ахмат сила, церковь, патриарх, голованова, монастыре"
7,1129,спорт,"спорта, спортсменов, спорт, сборной, игр, спортивных, спортсмены, спортивной, соревнованиях, мок, олимпийских, олимпиады"
8,1055,искусственный интеллект / технологии,"искусственного, искусственного интеллекта, интеллекта, роскосмоса, мс-, технологий, интеллект, су-, технологии, науки, искусственный, искусственный интеллект"
9,911,образование / школы / студенты,"первых, образования, учителей, студентов, учителя, егэ, школы, образование, ребята, минобрнауки, школьников, педагогов"


In [32]:
#число публикаций по темам в размеченной выборке 100k
topic_counts = (
    doc_topic_sentiment
    .groupby(["topic_id", "topic_name"], as_index=False)
    .agg(Число_публикаций=("doc_id", "count"))
    .sort_values("Число_публикаций", ascending=False)
)

topic_counts["Доля"] = topic_counts["Число_публикаций"] / topic_counts["Число_публикаций"].sum()
topic_counts = topic_counts.rename(columns={"topic_id": "№ темы", "topic_name": "Название темы"})

word_table(topic_counts.head(10), "Таблица. Топ-10 тем по числу публикаций в выборке 100k")

№ темы,Название темы,Число_публикаций,Доля
-1,outlier,36383,0.364
0,международная политика / Украина / США,30939,0.309
1,боевые действия / фронт / военные сводки,11467,0.115
2,экономика / рынок / энергетика,5688,0.057
3,ПВО / Курская область / ракетные угрозы,3343,0.033
4,культура / кино / медиа,2767,0.028
5,цифровая среда / мошенничество / платформы,2204,0.022
6,Украина / регионы / церковь,1371,0.014
7,спорт,1129,0.011
8,искусственный интеллект / технологии,1055,0.011


In [33]:
#топ тем без outlier-класса
topic_counts_no_outlier = topic_counts[topic_counts["№ темы"] != -1].copy()

word_table(topic_counts_no_outlier.head(10), "Таблица. Топ-10 содержательных тем по числу публикаций")

№ темы,Название темы,Число_публикаций,Доля
0,международная политика / Украина / США,30939,0.309
1,боевые действия / фронт / военные сводки,11467,0.115
2,экономика / рынок / энергетика,5688,0.057
3,ПВО / Курская область / ракетные угрозы,3343,0.033
4,культура / кино / медиа,2767,0.028
5,цифровая среда / мошенничество / платформы,2204,0.022
6,Украина / регионы / церковь,1371,0.014
7,спорт,1129,0.011
8,искусственный интеллект / технологии,1055,0.011
9,образование / школы / студенты,911,0.009


In [34]:
#считаем sentiment-распределение по темам
topic_sentiment = (
    doc_topic_sentiment
    .groupby(["topic_id", "topic_name"], as_index=False)
    .agg(
        Число_публикаций=("doc_id", "count"),
        Средний_sentiment=("sent", "mean"),
        Средняя_вероятность_negative=("sent_neg", "mean"),
        Средняя_вероятность_neutral=("sent_neu", "mean"),
        Средняя_вероятность_positive=("sent_pos", "mean")
    )
)

topic_sentiment = topic_sentiment.rename(columns={"topic_id": "№ темы", "topic_name": "Название темы"})
topic_sentiment = topic_sentiment.sort_values("Число_публикаций", ascending=False).reset_index(drop=True)

word_table(topic_sentiment.head(10), "Таблица. Sentiment-показатели крупнейших тем")

№ темы,Название темы,Число_публикаций,Средний_sentiment,Средняя_вероятность_negative,Средняя_вероятность_neutral,Средняя_вероятность_positive
-1,outlier,36383,-0.159,0.273,0.613,0.114
0,международная политика / Украина / США,30939,-0.196,0.275,0.647,0.078
1,боевые действия / фронт / военные сводки,11467,-0.131,0.233,0.666,0.102
2,экономика / рынок / энергетика,5688,0.086,0.152,0.609,0.238
3,ПВО / Курская область / ракетные угрозы,3343,-0.224,0.281,0.662,0.057
4,культура / кино / медиа,2767,0.024,0.204,0.567,0.229
5,цифровая среда / мошенничество / платформы,2204,-0.255,0.350,0.555,0.095
6,Украина / регионы / церковь,1371,-0.084,0.261,0.562,0.177
7,спорт,1129,0.117,0.183,0.516,0.301
8,искусственный интеллект / технологии,1055,0.126,0.110,0.654,0.236


In [35]:
#доли сентимент-классов по темам
topic_sentiment_shares = (
    doc_topic_sentiment
    .pivot_table(index=["topic_id", "topic_name"], columns="sent_label", values="doc_id", aggfunc="count", fill_value=0)
    .reset_index()
)

for col in ["negative", "neutral", "positive"]:
    if col not in topic_sentiment_shares.columns:
        topic_sentiment_shares[col] = 0

topic_sentiment_shares["Всего"] = topic_sentiment_shares[["negative", "neutral", "positive"]].sum(axis=1)
topic_sentiment_shares["Доля negative"] = topic_sentiment_shares["negative"] / topic_sentiment_shares["Всего"]
topic_sentiment_shares["Доля neutral"] = topic_sentiment_shares["neutral"] / topic_sentiment_shares["Всего"]
topic_sentiment_shares["Доля positive"] = topic_sentiment_shares["positive"] / topic_sentiment_shares["Всего"]

topic_sentiment_shares = topic_sentiment_shares.rename(columns={
    "topic_id": "№ темы",
    "topic_name": "Название темы",
    "negative": "Negative",
    "neutral": "Neutral",
    "positive": "Positive"
})

topic_sentiment_shares = topic_sentiment_shares.sort_values("Всего", ascending=False).reset_index(drop=True)

word_table(topic_sentiment_shares.head(10), "Таблица. Распределение sentiment-классов по крупнейшим темам")

№ темы,Название темы,Negative,Neutral,Positive,Всего,Доля negative,Доля neutral,Доля positive
-1,outlier,15947,15041,5395,36383,0.438,0.413,0.148
0,международная политика / Украина / США,13800,13908,3231,30939,0.446,0.450,0.104
1,боевые действия / фронт / военные сводки,4220,5654,1593,11467,0.368,0.493,0.139
2,экономика / рынок / энергетика,1444,2417,1827,5688,0.254,0.425,0.321
3,ПВО / Курская область / ракетные угрозы,1407,1759,177,3343,0.421,0.526,0.053
4,культура / кино / медиа,847,1105,815,2767,0.306,0.399,0.295
5,цифровая среда / мошенничество / платформы,1142,792,270,2204,0.518,0.359,0.123
6,Украина / регионы / церковь,540,526,305,1371,0.394,0.384,0.222
7,спорт,319,391,419,1129,0.283,0.346,0.371
8,искусственный интеллект / технологии,197,512,346,1055,0.187,0.485,0.328


In [36]:
#готовим таблицу тем с порогом по числу публикаций
topic_sentiment_filtered = topic_sentiment_shares[(topic_sentiment_shares["№ темы"] != -1) & (topic_sentiment_shares["Всего"] >= 100)].copy()

In [37]:
#топ-10 тем по доле negative
top_negative_topics = topic_sentiment_filtered.sort_values("Доля negative", ascending=False).head(10)
word_table(top_negative_topics, "Таблица. Топ-10 тем по доле negative")

№ темы,Название темы,Negative,Neutral,Positive,Всего,Доля negative,Доля neutral,Доля positive
11,энергетика / аварии / отключения,296,197,55,548,0.540,0.359,0.100
5,цифровая среда / мошенничество / платформы,1142,792,270,2204,0.518,0.359,0.123
15,погода / температура,78,48,29,155,0.503,0.310,0.187
14,алкоголь / вейпы / регулирование,136,109,31,276,0.493,0.395,0.112
0,международная политика / Украина / США,13800,13908,3231,30939,0.446,0.450,0.104
3,ПВО / Курская область / ракетные угрозы,1407,1759,177,3343,0.421,0.526,0.053
6,Украина / регионы / церковь,540,526,305,1371,0.394,0.384,0.222
13,семья / рождаемость / аборты,135,145,68,348,0.388,0.417,0.195
1,боевые действия / фронт / военные сводки,4220,5654,1593,11467,0.368,0.493,0.139
16,иноагенты / политические ограничения,41,73,4,118,0.347,0.619,0.034


In [38]:
#топ-10 тем по доле positive
top_positive_topics = topic_sentiment_filtered.sort_values("Доля positive", ascending=False).head(10)
word_table(top_positive_topics, "Таблица. Топ-10 тем по доле positive")

№ темы,Название темы,Negative,Neutral,Positive,Всего,Доля negative,Доля neutral,Доля positive
9,образование / школы / студенты,134,350,427,911,0.147,0.384,0.469
7,спорт,319,391,419,1129,0.283,0.346,0.371
10,здравоохранение / медицина,204,251,258,713,0.286,0.352,0.362
8,искусственный интеллект / технологии,197,512,346,1055,0.187,0.485,0.328
2,экономика / рынок / энергетика,1444,2417,1827,5688,0.254,0.425,0.321
4,культура / кино / медиа,847,1105,815,2767,0.306,0.399,0.295
12,авиация / аэропорты / ограничения,107,228,98,433,0.247,0.527,0.226
6,Украина / регионы / церковь,540,526,305,1371,0.394,0.384,0.222
13,семья / рождаемость / аборты,135,145,68,348,0.388,0.417,0.195
15,погода / температура,78,48,29,155,0.503,0.310,0.187


In [39]:
#топ-10 тем по доле neutral
top_neutral_topics = topic_sentiment_filtered.sort_values("Доля neutral", ascending=False).head(10)
word_table(top_neutral_topics, "Таблица. Топ-10 тем по доле neutral")

№ темы,Название темы,Negative,Neutral,Positive,Всего,Доля negative,Доля neutral,Доля positive
16,иноагенты / политические ограничения,41,73,4,118,0.347,0.619,0.034
12,авиация / аэропорты / ограничения,107,228,98,433,0.247,0.527,0.226
3,ПВО / Курская область / ракетные угрозы,1407,1759,177,3343,0.421,0.526,0.053
1,боевые действия / фронт / военные сводки,4220,5654,1593,11467,0.368,0.493,0.139
8,искусственный интеллект / технологии,197,512,346,1055,0.187,0.485,0.328
0,международная политика / Украина / США,13800,13908,3231,30939,0.446,0.450,0.104
2,экономика / рынок / энергетика,1444,2417,1827,5688,0.254,0.425,0.321
13,семья / рождаемость / аборты,135,145,68,348,0.388,0.417,0.195
4,культура / кино / медиа,847,1105,815,2767,0.306,0.399,0.295
14,алкоголь / вейпы / регулирование,136,109,31,276,0.493,0.395,0.112


In [40]:
#таблица контрольных показателей по каналам
controls_clean = controls.copy()
controls_clean = controls_clean.rename(columns={
    "channel_ref": "Канал",
    "posts": "Число публикаций",
    "topic_diversity": "Тематическое разнообразие",
    "negative_share": "Доля negative"
})

word_table(controls_clean.sort_values("Число публикаций", ascending=False).head(10), "Таблица. Топ-10 каналов по числу публикаций")

Канал,Число публикаций,Тематическое разнообразие,Доля negative
@ejdailyru,13078,0.537,0.337
@SolovievLive,11065,0.546,0.370
@boris_rozhin,6134,0.427,0.395
@dimsmirnov175,5296,0.690,0.407
@RVvoenkor,3491,0.418,0.516
@dmitrynikotin,3005,0.510,0.418
@mod_russia,2319,0.209,0.193
@ASGasparyan,2241,0.572,0.446
@Putin_tramp_mobilizaciya_migrant,1894,0.373,0.438
@rybar,1852,0.384,0.437


In [41]:
#топ-10 каналов по тематическому разнообразию
top_diverse_channels = controls_clean.sort_values("Тематическое разнообразие", ascending=False).head(10)
word_table(top_diverse_channels, "Таблица. Топ-10 каналов по тематическому разнообразию")

Канал,Число публикаций,Тематическое разнообразие,Доля negative
@vv_volodin,62,0.701,0.355
@dimsmirnov175,5296,0.690,0.407
@razvozhaev,560,0.682,0.150
@stalin_gulag,149,0.681,0.732
@Hinshtein,400,0.674,0.307
@Sandymustache,478,0.649,0.552
@Yuri_Slusar,178,0.641,0.258
@obrazbuduschego2,769,0.638,0.571
@MedvedevVesti,615,0.634,0.618
@rustroyka1945,563,0.618,0.535


In [42]:
#топ-10 каналов по доле negative
top_negative_channels = controls_clean.sort_values("Доля negative", ascending=False).head(10)
word_table(top_negative_channels, "Таблица. Топ-10 каналов по доле negative")

Канал,Число публикаций,Тематическое разнообразие,Доля negative
@mnogonazi,879,0.363,0.736
@stalin_gulag,149,0.681,0.732
@aavst2022,234,0.567,0.714
@slvn_pomet,942,0.487,0.707
@rlz_the_kraken,1400,0.555,0.667
@oabch,196,0.559,0.648
@khodorkovski,82,0.259,0.646
@eschulmann,190,0.495,0.632
@MedvedevVesti,615,0.634,0.618
@golosmordora1,1681,0.484,0.615


In [43]:
#собираем краткую сводку по размеченной выборке
corpus_summary = pd.DataFrame({
    "Показатель": [
        "Число публикаций",
        "Число каналов",
        "Число содержательных тем",
        "Число outlier-публикаций",
        "Доля outlier-публикаций",
        "Начало периода",
        "Конец периода"
    ],
    "Значение": [
        len(doc_topic_sentiment),
        doc_topic_sentiment["channel_ref"].nunique(),
        doc_topic_sentiment.loc[doc_topic_sentiment["topic_id"] != -1, "topic_id"].nunique(),
        (doc_topic_sentiment["topic_id"] == -1).sum(),
        (doc_topic_sentiment["topic_id"] == -1).mean(),
        doc_topic_sentiment["date"].min(),
        doc_topic_sentiment["date"].max()
    ]
})

word_table(corpus_summary, "Таблица. Сводка по размеченной выборке 100k")

Показатель,Значение
Число публикаций,100000
Число каналов,100
Число содержательных тем,19
Число outlier-публикаций,36383
Доля outlier-публикаций,0.364
Начало периода,2024-01-13 14:10:37+00:00
Конец периода,2026-01-12 16:52:06+00:00


#### 4. Распределение топиков в каналах по сентименту

In [44]:
topic_cols = [c for c in C.columns if c.startswith("topic_")]

def matrix_to_long(df, value_name):
    out = df.melt(id_vars="channel_ref", value_vars=topic_cols, var_name="topic_col", value_name=value_name)
    out["topic_id"] = out["topic_col"].str.replace("topic_", "", regex=False).astype(int)
    return out.drop(columns="topic_col")

C_long = matrix_to_long(C, "n_posts")
P_long = matrix_to_long(P, "topic_share")
S_long = matrix_to_long(S, "avg_sentiment")
A_pos_long = matrix_to_long(A_pos, "n_positive")
A_neg_long = matrix_to_long(A_neg, "n_negative")
A_neu_long = matrix_to_long(A_neu, "n_neutral")

matrix_long = C_long.merge(P_long, on=["channel_ref", "topic_id"])
matrix_long = matrix_long.merge(S_long, on=["channel_ref", "topic_id"])
matrix_long = matrix_long.merge(A_pos_long, on=["channel_ref", "topic_id"])
matrix_long = matrix_long.merge(A_neg_long, on=["channel_ref", "topic_id"])
matrix_long = matrix_long.merge(A_neu_long, on=["channel_ref", "topic_id"])

topic_names = top_words[["topic_id", "topic_name_manual"]].rename(columns={"topic_name_manual": "topic_name"})
matrix_long = matrix_long.merge(topic_names, on="topic_id", how="left")

In [45]:
#определяем главную тему каждого канала по максимальной доле
main_topic_by_channel = (
    matrix_long
    .sort_values(["channel_ref", "topic_share"], ascending=[True, False])
    .groupby("channel_ref")
    .head(1)
    .reset_index(drop=True)
)

main_topic_by_channel = main_topic_by_channel[["channel_ref", "topic_id", "topic_name", "n_posts", "topic_share", "avg_sentiment"]]
main_topic_by_channel = main_topic_by_channel.rename(columns={
    "channel_ref": "Канал",
    "topic_id": "Главная тема",
    "topic_name": "Название темы",
    "n_posts": "Число публикаций в теме",
    "topic_share": "Доля главной темы",
    "avg_sentiment": "Средний sentiment по теме"
})

word_table(main_topic_by_channel.sort_values("Доля главной темы", ascending=False).head(10), "Таблица. Топ-10 каналов с наиболее выраженной тематической специализацией")

Канал,Главная тема,Название темы,Число публикаций в теме,Доля главной темы,Средний sentiment по теме
@mod_russia,1,боевые действия / фронт / военные сводки,1549.000,0.871,-0.012
@medvedev_telegram,0,международная политика / Украина / США,10.000,0.833,-0.177
@gubernator_46,3,ПВО / Курская область / ракетные угрозы,736.000,0.812,-0.044
@skabeeva,0,международная политика / Украина / США,685.000,0.800,-0.172
@MariaVladimirovnaZakharova,0,международная политика / Украина / США,172.000,0.793,-0.093
@ironlogica,0,международная политика / Украина / США,223.000,0.782,-0.128
@Taynaya_kantselyariya,0,международная политика / Украина / США,242.000,0.776,-0.155
@obzorpolita,0,международная политика / Украина / США,47.000,0.770,-0.031
@khodorkovski,0,международная политика / Украина / США,33.000,0.767,-0.459
@EvPanina,0,международная политика / Украина / США,263.000,0.762,-0.130


In [46]:
#считаем доли sentiment-классов внутри каждой пары канал-тема
min_posts = 10

matrix_long["sent_total"] = matrix_long["n_positive"] + matrix_long["n_negative"] + matrix_long["n_neutral"]
matrix_long["negative_share_topic"] = np.where(matrix_long["sent_total"] > 0, matrix_long["n_negative"] / matrix_long["sent_total"], np.nan)
matrix_long["neutral_share_topic"] = np.where(matrix_long["sent_total"] > 0, matrix_long["n_neutral"] / matrix_long["sent_total"], np.nan)
matrix_long["positive_share_topic"] = np.where(matrix_long["sent_total"] > 0, matrix_long["n_positive"] / matrix_long["sent_total"], np.nan)

In [47]:
#выводим пары канал-тема с наибольшей долей negative
top_channel_topic_negative_share = (
    matrix_long[matrix_long["n_posts"] >= min_posts]
    .sort_values("negative_share_topic", ascending=False)
    .head(10)
)

top_channel_topic_negative_share = top_channel_topic_negative_share[["channel_ref", "topic_id", "topic_name", "n_posts", "negative_share_topic", "neutral_share_topic", "positive_share_topic", "avg_sentiment"]]
top_channel_topic_negative_share = top_channel_topic_negative_share.rename(columns={
    "channel_ref": "Канал",
    "topic_id": "№ темы",
    "topic_name": "Название темы",
    "n_posts": "Число публикаций",
    "negative_share_topic": "Доля negative",
    "neutral_share_topic": "Доля neutral",
    "positive_share_topic": "Доля positive",
    "avg_sentiment": "Средний sentiment"
})

word_table(top_channel_topic_negative_share, "Таблица. Топ-10 пар канал–тема по доле negative")

Канал,№ темы,Название темы,Число публикаций,Доля negative,Доля neutral,Доля positive,Средний sentiment
@boris_rozhin,6,Украина / регионы / церковь,56.000,0.875,0.107,0.018,-0.733
@voenkorKotenok,6,Украина / регионы / церковь,14.000,0.857,0.071,0.071,-0.416
@mnogonazi,3,ПВО / Курская область / ракетные угрозы,27.000,0.852,0.111,0.037,-0.491
@golosmordora1,7,спорт,13.000,0.846,0.077,0.077,-0.388
@rlz_the_kraken,11,энергетика / аварии / отключения,13.000,0.846,0.154,0.000,-0.547
@MedvedevVesti,5,цифровая среда / мошенничество / платформы,13.000,0.846,0.000,0.154,-0.442
@mnogonazi,2,экономика / рынок / энергетика,11.000,0.818,0.182,0.000,-0.486
@mos_sobyanin,1,боевые действия / фронт / военные сводки,63.000,0.810,0.079,0.111,-0.512
@stalin_gulag,0,международная политика / Украина / США,31.000,0.806,0.194,0.000,-0.511
@slvn_pomet,3,ПВО / Курская область / ракетные угрозы,10.000,0.800,0.100,0.100,-0.549


In [48]:
#выводим пары канал-тема с наибольшей долей positive
top_channel_topic_positive_share = (
    matrix_long[matrix_long["n_posts"] >= min_posts]
    .sort_values("positive_share_topic", ascending=False)
    .head(10)
)

top_channel_topic_positive_share = top_channel_topic_positive_share[["channel_ref", "topic_id", "topic_name", "n_posts", "negative_share_topic", "neutral_share_topic", "positive_share_topic", "avg_sentiment"]]
top_channel_topic_positive_share = top_channel_topic_positive_share.rename(columns={
    "channel_ref": "Канал",
    "topic_id": "№ темы",
    "topic_name": "Название темы",
    "n_posts": "Число публикаций",
    "negative_share_topic": "Доля negative",
    "neutral_share_topic": "Доля neutral",
    "positive_share_topic": "Доля positive",
    "avg_sentiment": "Средний sentiment"
})

word_table(top_channel_topic_positive_share, "Таблица. Топ-10 пар канал–тема по доле positive")

Канал,№ темы,Название темы,Число публикаций,Доля negative,Доля neutral,Доля positive,Средний sentiment
@mos_sobyanin,10,здравоохранение / медицина,24.000,0.000,0.042,0.958,0.801
@government_rus,12,авиация / аэропорты / ограничения,21.000,0.000,0.048,0.952,0.819
@mos_sobyanin,7,спорт,20.000,0.000,0.050,0.950,0.766
@mos_sobyanin,9,образование / школы / студенты,16.000,0.000,0.062,0.938,0.782
@obshina_ru,18,пожертвования / поддержка организаций,63.000,0.032,0.032,0.937,0.811
@RKadyrov_95,7,спорт,15.000,0.000,0.067,0.933,0.883
@mkhusnullin,2,экономика / рынок / энергетика,80.000,0.000,0.125,0.875,0.747
@Yuri_Slusar,9,образование / школы / студенты,14.000,0.000,0.143,0.857,0.615
@vvgladkov,10,здравоохранение / медицина,11.000,0.091,0.091,0.818,0.676
@government_rus,7,спорт,25.000,0.000,0.200,0.800,0.627


#### 5. Таблица-саммари

In [49]:
#готовим принадлежность каналов к кластерам stance-сети
channel_clusters = (
    stance_nodes[stance_nodes["node_type"] == "channel"]
    [["node_name", "modularity_class"]]
    .rename(columns={
        "node_name": "channel_ref",
        "modularity_class": "Класс модулярности"
    })
)

#определяем главную тему каждого канала по максимальной доле
main_topic_by_channel_full = (
    matrix_long[matrix_long["n_posts"] > 0]
    .sort_values(["channel_ref", "topic_share"], ascending=[True, False])
    .groupby("channel_ref")
    .head(1)
    .reset_index(drop=True)
)

main_topic_by_channel_full = main_topic_by_channel_full[[
    "channel_ref",
    "topic_id",
    "topic_name",
    "n_posts",
    "topic_share",
    "avg_sentiment",
    "negative_share_topic",
    "neutral_share_topic",
    "positive_share_topic"
]]

main_topic_by_channel_full = main_topic_by_channel_full.rename(columns={
    "topic_id": "Главная тема",
    "topic_name": "Название главной темы",
    "n_posts": "Публикаций в главной теме",
    "topic_share": "Доля главной темы",
    "avg_sentiment": "Средний sentiment главной темы",
    "negative_share_topic": "Доля negative в главной теме",
    "neutral_share_topic": "Доля neutral в главной теме",
    "positive_share_topic": "Доля positive в главной теме"
})

#собираем топ-3 темы каждого канала
top_topics_by_channel = (
    matrix_long[matrix_long["n_posts"] > 0]
    .sort_values(["channel_ref", "topic_share"], ascending=[True, False])
    .groupby("channel_ref")
    .head(3)
    .copy()
)

top_topics_by_channel["topic_short"] = (
    top_topics_by_channel["topic_id"].astype(str)
    + ". "
    + top_topics_by_channel["topic_name"].astype(str)
    + " ("
    + (top_topics_by_channel["topic_share"] * 100).round(1).astype(str)
    + "%)"
)

top_topics_by_channel = (
    top_topics_by_channel
    .groupby("channel_ref", as_index=False)
    .agg(
        Топ_3_темы=("topic_short", " | ".join)
    )
)

#готовим контрольные показатели каналов
controls_for_table = controls.copy()

controls_for_table = controls_for_table.rename(columns={
    "channel_ref": "channel_ref",
    "posts": "Число публикаций",
    "topic_diversity": "Тематическое разнообразие",
    "negative_share": "Доля negative"
})

#собираем итоговую таблицу
channel_cluster_topic_sentiment = (
    controls_for_table
    .merge(channel_clusters, on="channel_ref", how="left")
    .merge(main_topic_by_channel_full, on="channel_ref", how="left")
    .merge(top_topics_by_channel, on="channel_ref", how="left")
    .rename(columns={"channel_ref": "Канал"})
    .sort_values(["Класс модулярности", "Число публикаций"], ascending=[True, False])
    .reset_index(drop=True)
)

word_table(
    channel_cluster_topic_sentiment.head(30),
    "Таблица. Каналы, кластеры, главные темы и sentiment",
    precision=3
)

Канал,Число публикаций,Тематическое разнообразие,Доля negative,Класс модулярности,Главная тема,Название главной темы,Публикаций в главной теме,Доля главной темы,Средний sentiment главной темы,Доля negative в главной теме,Доля neutral в главной теме,Доля positive в главной теме,Топ_3_темы
@ejdailyru,13078,0.537,0.337,0.000,0.000,международная политика / Украина / США,4926.000,0.580,-0.153,0.352,0.575,0.073,0. международная политика / Украина / США (58.0%) | 2. экономика / рынок / энергетика (15.3%) | 5. цифровая среда / мошенничество / платформы (5.7%)
@SolovievLive,11065,0.546,0.370,0.000,0.000,международная политика / Украина / США,4019.000,0.527,-0.161,0.384,0.506,0.110,0. международная политика / Украина / США (52.7%) | 1. боевые действия / фронт / военные сводки (18.5%) | 3. ПВО / Курская область / ракетные угрозы (10.2%)
@dimsmirnov175,5296,0.690,0.407,0.000,0.000,международная политика / Украина / США,1479.000,0.430,-0.123,0.421,0.421,0.159,0. международная политика / Украина / США (43.0%) | 2. экономика / рынок / энергетика (13.3%) | 5. цифровая среда / мошенничество / платформы (8.2%)
@dmitrynikotin,3005,0.510,0.418,0.000,0.000,международная политика / Украина / США,1214.000,0.633,-0.178,0.439,0.419,0.142,0. международная политика / Украина / США (63.3%) | 1. боевые действия / фронт / военные сводки (7.5%) | 2. экономика / рынок / энергетика (6.4%)
@Putin_tramp_mobilizaciya_migrant,1894,0.373,0.438,0.000,0.000,международная политика / Украина / США,910.000,0.761,-0.167,0.373,0.574,0.054,0. международная политика / Украина / США (76.1%) | 2. экономика / рынок / энергетика (4.5%) | 4. культура / кино / медиа (4.1%)
@mig41,1552,0.606,0.472,0.000,0.000,международная политика / Украина / США,483.000,0.515,-0.214,0.474,0.416,0.110,0. международная политика / Украина / США (51.5%) | 1. боевые действия / фронт / военные сводки (12.6%) | 8. искусственный интеллект / технологии (6.1%)
@skabeeva,1541,0.315,0.463,0.000,0.000,международная политика / Украина / США,685.000,0.800,-0.172,0.453,0.429,0.118,0. международная политика / Украина / США (80.0%) | 1. боевые действия / фронт / военные сводки (7.2%) | 7. спорт (2.2%)
@kstati_p,1439,0.555,0.258,0.000,0.000,международная политика / Украина / США,481.000,0.516,-0.101,0.274,0.642,0.083,0. международная политика / Украина / США (51.6%) | 2. экономика / рынок / энергетика (20.7%) | 5. цифровая среда / мошенничество / платформы (7.3%)
@protiktok,1408,0.379,0.221,0.000,0.000,международная политика / Украина / США,737.000,0.728,-0.062,0.210,0.689,0.100,0. международная политика / Украина / США (72.8%) | 2. экономика / рынок / энергетика (11.3%) | 17. региональная политика / ПФО (4.9%)
@polit_inform,1240,0.449,0.381,0.000,0.000,международная политика / Украина / США,531.000,0.670,-0.135,0.335,0.578,0.087,0. международная политика / Украина / США (67.0%) | 1. боевые действия / фронт / военные сводки (8.7%) | 2. экономика / рынок / энергетика (8.2%)


In [50]:
#сохраняем итоговую таблицу для дальнейшей интерпретации кластеров
channel_cluster_topic_sentiment.to_excel(
    data_dir / "channel_cluster_topic_sentiment.xlsx",
    index=False
)

channel_cluster_topic_sentiment.to_csv(
    data_dir / "channel_cluster_topic_sentiment.csv",
    index=False,
    encoding="utf-8-sig"
)

print("таблица сохранена")

таблица сохранена
